<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

### <font color='#BFD72F'>**Methodology** </font> <a class="anchor" id='top'></a> 

- [1. Introduction](#1) 
- [2. Import Libraries](#2) 
- [3. Create Metadata](#3)
- [4. Import Dataset](#4)  
- [5. Model and Assessment](#5) 
    - [5.1 Linear Regression](#5_1)
        - [5.1.1 Ordinary Least Squares (OLS)](#5_1_1)
        - [5.1.2 Ridge Regression](#5_1_2)
        - [5.1.3 Lasso Regression](#5_1_3)
        - [5.1.4 Elastic Net Regression](#5_1_4)
        - [5.1.5 Random Forest](#5_1_5)
    - [5.2 Model Comparison](#5_2)
    - [5.3 Test Models](#5_3)
- [6. Save the Results to Kaggle](#6) 
- [7. End of the Notebook](#7) 

<a class="anchor" id="1">

# **1. Introduction**

[Back to TOP](#TOP)
</a>

Here we will create models to predict the used car prices. We will create several models and later analyse their performances and choose the one that best predicts the target feature based on the MAE metric. The results here represent our final models with the thresholds and features defined in the "03_Feature_Selection" file that yield the best results we could.

<a class="anchor" id="2">

# **2. Import libraries**

[Back to TOP](#TOP)
</a>

The following libraries will help us develop the analyses and model for this project. Pandas and Numpy, provide the efficient tools for data manipulation, cleaning and numerical computations. Matplotlib and Seaborn are used to create clear and informative visualizations. Finally, Scikit-learn offers a range of Machine Learning tools for model training, spliting the data and evaluate model performance. 

In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import os


from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

#KNN
from sklearn.neighbors import KNeighborsRegressor

# Random Forest
from sklearn.ensemble import RandomForestRegressor

#Neural Network
from sklearn.neural_network import MLPRegressor


#Model evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, mean_absolute_percentage_error
import statsmodels.api as sm

# Load Created Functions
from visualizations import *
from data_preprocessing import *
from model_and_assessment import *

# Set random seed for reproducibility
np.random.seed(40111) 

<a class="anchor" id="3">

# **3. Create Metadata**

[Back to TOP](#TOP)
</a>

Understanding the features helps interpret the data correctly and supports subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



<a class="anchor" id="4">

# **4. Import Dataset**

[Back to TOP](#top)
</a>

In this section, we load the preprocessed datasets that were previously saved after the data preprocessing steps. These datasets are ready for modelling and to get results.

In [3]:
# Define relative path to the preprocessed data folder (outside "notebooks/")
data_path = "../data_feature_selected/"

# Load preprocessed datasets
X_train = pd.read_csv(f"{data_path}X_train_final.csv", index_col=0)
X_val   = pd.read_csv(f"{data_path}X_val_final.csv", index_col=0)
test    = pd.read_csv(f"{data_path}test_final.csv", index_col=0)

# Load target variables and squeeze to convert DataFrame to Series since they have only one column
y_train = pd.read_csv(f"{data_path}y_train.csv", index_col=0).squeeze()
y_val   = pd.read_csv(f"{data_path}y_val.csv", index_col=0).squeeze()

<a class="anchor" id="5">

# **5. Model and Assessment**

[Back to TOP](#TOP)
</a>

In this notebook, we focus on training and evaluating different regression models to predict car resale prices. We start with a simple model to establish a baseline and progressively explore more complex models. The purpose of this notebook section is to create a structured workflow that allows us to train models efficiently, evaluate their performance, and compare predictions with actual values.

In [4]:
# Combine train and validation datasets
X_combined = np.concatenate([X_train, X_val], axis=0)
y_combined = np.concatenate([y_train, y_val], axis=0)

# Create a test fold index (-1 for train, 0 for validation)
test_fold = [-1] * len(X_train) + [0] * len(X_val)

print('Test fold: ', len(test_fold))
print('X_combined: ', len(X_combined))
print('y_combined: ', len(y_combined))

# Define the PredefinedSplit
ps = PredefinedSplit(test_fold=test_fold) # aqui diz que os dados de treino são os que têm label -1 e os de validação são os que têm label 0

Test fold:  75973
X_combined:  75973
y_combined:  75973


### Create a Sample to Test Models

In [5]:
X_train_s = X_train.sample(1000, random_state=42)
y_train_s = y_train.loc[X_train_s.index]
X_val_s = X_val.sample(500, random_state=42)
y_val_s = y_val.loc[X_val_s.index]

# Recriar combined sample
sample_X_combined = np.concatenate([X_train_s, X_val_s])
sample_y_combined = np.concatenate([y_train_s, y_val_s])

# Novo fold correto!
sample_test_fold = [-1] * len(X_train_s) + [0] * len(X_val_s)

ps_sample = PredefinedSplit(sample_test_fold)

## Apply Predefined Split with KNN

In [ ]:
model_KNN = KNeighborsRegressor()
param_grid_knn = {
    "n_neighbors": [10, 20, 30, 40, 50, 80, 100],              # 3..80
    "weights": ["uniform", "distance"],
    "p": [1, 2],                                # Manhattan vs Euclidiana
    "leaf_size": [2, 5, 8, 12, 17, 25, 30, 40, 50, 60, 70, 80, 90, 100],
    "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
}

scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

## **Primeiro Teste do KNN com uma samples para perceber se o código está correto e faz sentido**

In [6]:
rsCV_KNN = RandomizedSearchCV(model_KNN, param_grid_knn, n_iter=10, scoring=scoring, refit='mae', verbose=1, cv=ps_sample, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_KNN.fit(sample_X_combined, sample_y_combined)

evaluate_model(rsCV_KNN)

Fitting 1 folds for each of 10 candidates, totalling 10 fits
Train R2=1.000 | Val R2=0.784 | Train MAE=-0.0 | Val MAE=2904.3 | Gap R2=0.216 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 50, 'leaf_size': 5, 'algorithm': 'brute'}
Train R2=1.000 | Val R2=0.747 | Train MAE=-0.0 | Val MAE=3109.1 | Gap R2=0.253 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 80, 'leaf_size': 12, 'algorithm': 'ball_tree'}
Train R2=1.000 | Val R2=0.807 | Train MAE=-0.0 | Val MAE=2786.9 | Gap R2=0.193 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 30, 'leaf_size': 50, 'algorithm': 'auto'}
Train R2=1.000 | Val R2=0.807 | Train MAE=-0.0 | Val MAE=2786.9 | Gap R2=0.193 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 30, 'leaf_size': 60, 'algorithm': 'kd_tree'}
Train R2=1.000 | Val R2=0.858 | Train MAE=-0.0 | Val MAE=2519.8 | Gap R2=0.142 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 10, 'leaf_size': 2, 'algorithm': 'auto'}
Train R2=0.773 | Val R2=0.750 | Train MAE=3094.

## **Testar Novamente KNN com o dataset todo**

In [ ]:
rsCV_KNN = RandomizedSearchCV(model_KNN, param_grid_knn, n_iter=10, scoring=scoring, refit='mae', verbose=1, cv=ps, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_KNN.fit(X_combined, y_combined)

evaluate_model(rsCV_KNN)

Fitting 1 folds for each of 15 candidates, totalling 15 fits
Train=0.876 | Val=0.872 | Gap=0.004 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 40, 'leaf_size': 25, 'algorithm': 'auto'}
Train=0.999 | Val=0.878 | Gap=0.121 | Params={'weights': 'distance', 'p': 2, 'n_neighbors': 100, 'leaf_size': 90, 'algorithm': 'kd_tree'}
Train=0.912 | Val=0.891 | Gap=0.021 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 10, 'leaf_size': 2, 'algorithm': 'ball_tree'}
Train=0.999 | Val=0.888 | Gap=0.111 | Params={'weights': 'distance', 'p': 2, 'n_neighbors': 50, 'leaf_size': 2, 'algorithm': 'brute'}
Train=0.884 | Val=0.878 | Gap=0.006 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 30, 'leaf_size': 2, 'algorithm': 'auto'}
Train=0.894 | Val=0.889 | Gap=0.005 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 30, 'leaf_size': 8, 'algorithm': 'ball_tree'}
Train=0.871 | Val=0.869 | Gap=0.003 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 50, 'leaf_size': 40, 'algorithm': 'br

KeyError: 'mean_test_MAE'

Ao que parece se usarmos 'distance' overfitta sempre. Agora vou testar o mesmo modelo mas tirando 'distance' como opção

In [ ]:
model_KNN = KNeighborsRegressor()
param_grid_knn = {
    "n_neighbors": [10, 20, 30, 40, 50, 80, 100],              # 3..80
    "weights": ["uniform"],
    "p": [1, 2],                                # Manhattan vs Euclidiana
    "leaf_size": [2, 5, 8, 12, 17, 25, 30, 40, 50, 60, 70, 80, 90, 100],
    "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
}

scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

In [ ]:
rsCV_KNN = RandomizedSearchCV(model_KNN, param_grid_knn, n_iter=15, scoring=scoring, refit='mae', verbose=1, cv=ps, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_KNN.fit(X_combined, y_combined)

evaluate_model(rsCV_KNN)

Fitting 1 folds for each of 15 candidates, totalling 15 fits
Train R2=0.894 | Val R2=0.883 | Train MAE=1797.5 | Val MAE=1920.9 | Gap R2=0.011 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 20, 'leaf_size': 30, 'algorithm': 'ball_tree'}
Train R2=0.883 | Val R2=0.879 | Train MAE=1918.7 | Val MAE=1979.0 | Gap R2=0.003 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 50, 'leaf_size': 5, 'algorithm': 'brute'}
Train R2=0.919 | Val R2=0.900 | Train MAE=1550.9 | Val MAE=1752.4 | Gap R2=0.019 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 10, 'leaf_size': 5, 'algorithm': 'brute'}
Train R2=0.867 | Val R2=0.864 | Train MAE=2082.2 | Val MAE=2125.5 | Gap R2=0.003 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 100, 'leaf_size': 50, 'algorithm': 'ball_tree'}
Train R2=0.883 | Val R2=0.879 | Train MAE=1918.7 | Val MAE=1979.0 | Gap R2=0.003 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 50, 'leaf_size': 5, 'algorithm': 'ball_tree'}
Train R2=0.884 | Val R2=0.878 | Tr

## **Random Forest**

In [8]:
model_rf = RandomForestRegressor()

param_grid_rf = {
    "n_estimators": [200, 400, 600],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True]
}

scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

In [9]:
rsCV_rf = RandomizedSearchCV(model_rf, param_grid_rf, n_iter=15, scoring=scoring, refit='mae', verbose=1, cv=ps_sample, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_rf.fit(sample_X_combined, sample_y_combined)

evaluate_model(rsCV_rf)

Fitting 1 folds for each of 15 candidates, totalling 15 fits
Train R2=0.979 | Val R2=0.825 | Train MAE=905.6 | Val MAE=2535.5 | Gap R2=0.154 | Params={'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'bootstrap': True}
Train R2=0.924 | Val R2=0.813 | Train MAE=1762.6 | Val MAE=2623.2 | Gap R2=0.111 | Params={'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 10, 'bootstrap': True}
Train R2=0.920 | Val R2=0.814 | Train MAE=1776.4 | Val MAE=2608.7 | Gap R2=0.106 | Params={'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 20, 'bootstrap': True}
Train R2=0.937 | Val R2=0.812 | Train MAE=1650.0 | Val MAE=2651.1 | Gap R2=0.125 | Params={'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10, 'bootstrap': True}
Train R2=0.979 | Val R2=0.824 | Train MAE=914.1 | Val MAE=2540.9 | Gap R2=0.155 | Params={'n_estimators': 600, 'min_samples_split': 2, 'min_samples_leaf': 1

In [ ]:
best_model = rsCV_rf.best_estimator_
preds = best_model.predict(X_val)
mean_absolute_error(y_val, preds)   

#este warning tb aparece na aula se tiramos a library dos warnings. Não sei se temos um erro ou não

/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


2353.173439727981

**Todo o Dataset**

In [13]:
rsCV_rf = RandomizedSearchCV(model_rf, param_grid_rf, n_iter=15, scoring=scoring, refit='mae', verbose=1, cv=ps, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_rf.fit(X_combined, y_combined)

evaluate_model(rsCV_rf)

Fitting 1 folds for each of 15 candidates, totalling 15 fits
Train R2=0.960 | Val R2=0.917 | Train MAE=1082.0 | Val MAE=1608.7 | Gap R2=0.043 | Params={'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': None, 'bootstrap': True}
Train R2=0.910 | Val R2=0.891 | Train MAE=1901.4 | Val MAE=2028.2 | Gap R2=0.018 | Params={'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 10, 'bootstrap': True}
Train R2=0.960 | Val R2=0.917 | Train MAE=1083.4 | Val MAE=1611.2 | Gap R2=0.044 | Params={'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 30, 'bootstrap': True}
Train R2=0.959 | Val R2=0.917 | Train MAE=1115.8 | Val MAE=1606.6 | Gap R2=0.042 | Params={'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': True}
Train R2=0.952 | Val R2=0.915 | Train MAE=1182.4 | Val MAE=1625.4 | Gap R2=0.037 | Params={'n_estimators': 600, 'min_samples_split': 10, 'min_samples_leaf

In [14]:
preds = best_model.predict(test)

/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


## Apply the model to the test set

## Neural Networks

In [ ]:
model_nn = MLPRegressor()

param_grid_nn = {
    
    "hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 50)],
    "activation": ["relu", "tanh"],
    "solver": ["adam", "sgd"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate": ["constant", "adaptive"]
}

In [ ]:
rsCV_nn = RandomizedSearchCV(model_nn, param_grid_nn, n_iter=10, scoring=scoring, refit='mae', verbose=1, cv=ps_sample, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_nn.fit(sample_X_combined, sample_y_combined)

evaluate_model(rsCV_nn)

Fitting 1 folds for each of 10 candidates, totalling 10 fits


/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/s

Train R2=-2.626 | Val R2=-2.474 | Train MAE=17085.3 | Val MAE=16741.5 | Gap R2=-0.152 | Params={'solver': 'adam', 'learning_rate': 'constant', 'hidden_layer_sizes': (100,), 'alpha': 0.0001, 'activation': 'tanh'}
Train R2=-2.625 | Val R2=-2.473 | Train MAE=17082.5 | Val MAE=16738.8 | Gap R2=-0.152 | Params={'solver': 'adam', 'learning_rate': 'adaptive', 'hidden_layer_sizes': (100,), 'alpha': 0.01, 'activation': 'tanh'}
Train R2=0.160 | Val R2=0.141 | Train MAE=6534.4 | Val MAE=6432.4 | Gap R2=0.019 | Params={'solver': 'sgd', 'learning_rate': 'constant', 'hidden_layer_sizes': (100, 50), 'alpha': 0.01, 'activation': 'tanh'}
Train R2=0.530 | Val R2=0.481 | Train MAE=4811.0 | Val MAE=5038.8 | Gap R2=0.049 | Params={'solver': 'sgd', 'learning_rate': 'adaptive', 'hidden_layer_sizes': (100, 50), 'alpha': 0.001, 'activation': 'tanh'}
Train R2=-197325800973678196977343260929682682916740848627131635856311181153572360922031298534322372928448740062438704863029737156550903814209212311256488426391238

<a class="anchor" id="6">

# **6. Save results to Kaggle**

[Back to TOP](#TOP)
</a>

In [15]:
# Go one level up from the notebooks folder to reach the repo root
selected_dir = "../results/kaggle_submissions/"
os.makedirs(selected_dir, exist_ok=True)

In [17]:
# Extract carID
car_ids = test.index.values

# Create DataFrame with best model predictions
best_model_df = pd.DataFrame({
    'carID': car_ids,
    'price': preds
})

# Save predictions to CSV
best_model_df.to_csv(f"{selected_dir}/predictions.csv", index=False)


<a class="anchor" id="7">

# **7. End of the Notebook**

[Back to TOP](#TOP)
</a>

From the modeling and assessment stage, we developed and evaluated several predictive models to estimate car prices based on the processed features. Different algorithms were trained, and their performance was compared using appropriate evaluation metrics to identify the most effective approach. The analysis of results provided valuable insights into the model’s predictive capabilities, highlighting strengths, limitations, and potential areas for improvement. This stage represents a crucial step in validating the overall workflow and ensuring that the final model delivers accurate and reliable predictions..